# Module 2 · Lesson 07: Resume Customizer

Apply prompt engineering to a real career task: **tailoring resume bullet points**
to match a specific job description. This exercise ties together zero-shot, few-shot,
and constraint-based prompting.

## What you will learn
1. **System prompt design** for domain-specific tasks
2. **Few-shot examples** to control output style
3. **Constraint prompting** for length and format control
4. Iterative prompt refinement workflow

In [1]:
import os, json
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown
 
load_dotenv(Path.cwd().parent / ".env")
 
from openai import OpenAI
client = OpenAI()
 
def ask(prompt, system=None, temperature=0.7, max_tokens=500):
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": prompt})
    r = client.chat.completions.create(
        model="gpt-4o-mini", messages=msgs,
        temperature=temperature, max_tokens=max_tokens
    )
    return r.choices[0].message.content
 
print("Ready")

Ready


---
## 1. Define the Input

Two inputs: the **job description** (target) and your **work experience** (source).

In [2]:
job_description = """Senior Python Developer - AI Team

We're looking for an experienced Python developer to join our AI team.
You' ll build production ML pipelines, optimize model serving infrastructure,
and collaborate with data scientists.

Requirements:
- 5+ years of Python experience
- Experience with FastAPI or Flask
- ML/AI deployment (Docker, Kubernetes)
- Strong testing practices (pytestm CI/CD)
- Experience with cloud platforms (AWS/GCP)
"""

work_experience = """- Worked on a web application using Django for 3 years
- Helped deploy models to production servers
- Wrote some unit tests for the backed
- Used AWS for hosting and S3 for storage
- Collaborated with the data team on data pipelines
- Built REST APIs for internal tools"""

---
## 2. Zero-Shot Approach

First attempt: just ask the model to rewrite, no examples.

In [ ]:
ZERO_SHOT_PROMPT = """You are senior hiring manager and resume expert.
Rewrite the candidate's experience bullets to better match the target job description.

Rules:
- Keep the same facts, but reframe using job's language
- Start each bullet with a strong action verb
- Include qualifiable results where possible
- Maximum 1 line per bullet point
- Do NOT fabricate experience
"""

zero_shot = ask(
    f"""Job description:
{job_description}
    
Original Experience:
{work_experience}

Rewritten bullets:""",
system = ZERO_SHOT_PROMPT,
temperature = 0.5
)

display(Markdown(f"### Zero-shot resutlt:\n\n{zero_shot}"))
# Worked on a web application using Django for 3 years

### Zero-shot resutlt:

- Developed and maintained production-grade web applications using Django for over 3 years, enhancing user experience and functionality.  
- Deployed machine learning models to production servers, ensuring high availability and performance.  
- Implemented robust unit testing practices using pytest, contributing to a reliable codebase.  
- Utilized AWS for cloud hosting and S3 for scalable storage solutions, optimizing resource management.  
- Collaborated with data scientists to design and optimize data pipelines, streamlining workflows and improving data accessibility.  
- Engineered RESTful APIs for internal tools, facilitating seamless integration and communication between services.  

---
## 3. Few-Shot Approach

Now let's provide **examples** of good rewrites to guide the style:

In [4]:
FEW_SHOT_SYSTEM = """You are a senior hiring manager and resume expert.
Rewrite experience bullets to match a target job description.

Here are examples of good rewrites:

BEFORE: "Helped with database stuff"
AFTER: "Optimized PostgreSQL query performance, reducing p95 latency by 40% acrross 3 production services"

BEFORE: "Made some APIs"
AFTER: "Architected and deployed RESTful APIs serving 50K+ daily requests using FastAPI with async patterns"
 
BEFORE: "Did testing"
AFTER: "Established comprehensive test suite with 92% coverage using pytest, integrated into CI/CD pipeline"

Rules:
- Match the STYLE of the examples above
- Use the job description's specific keywords
- Start each bullet with a STRONG action verb
- Add plausible metrics (but mark estimated ones with ~)
- Do NOT fabricate experience - only reframe existing facts"""

In [5]:
few_shot = ask(
    f"""Job description:
{job_description}
    
Original Experience:
{work_experience}

Rewritten bullets:""",
system = FEW_SHOT_SYSTEM,
temperature = 0.5
)

display(Markdown(f"### Few-shot resutlt:\n\n{few_shot}"))

### Few-shot resutlt:

- Developed and maintained a high-performance web application using Django, enhancing user experience and scalability over a 3-year period.

- Deployed machine learning models to production environments, optimizing model serving infrastructure to handle ~10K requests per hour.

- Established robust unit testing framework with 85% coverage using pytest, seamlessly integrated into CI/CD pipeline to ensure code quality.

- Leveraged AWS for hosting and utilized S3 for efficient data storage, improving retrieval times by ~30% across multiple applications.

- Collaborated closely with the data team to design and implement data pipelines, enhancing data processing efficiency by ~25%.

- Built and optimized RESTful APIs for internal tools, supporting over 1,000 internal users and streamlining workflows across teams.

---
## 4. Compare Results

In [7]:
compare = f""" ### Comparison

#### Original
{work_experience}

#### Zero-Shot Rewrite
{zero_shot}

#### Few-Shot Rewrite
{few_shot}
"""

display(Markdown(compare))

 ### Comparison

#### Original
- Worked on a web application using Django for 3 years
- Helped deploy models to production servers
- Wrote some unit tests for the backed
- Used AWS for hosting and S3 for storage
- Collaborated with the data team on data pipelines
- Built REST APIs for internal tools

#### Zero-Shot Rewrite
- Developed and maintained production-grade web applications using Django for over 3 years, enhancing user experience and functionality.  
- Deployed machine learning models to production servers, ensuring high availability and performance.  
- Implemented robust unit testing practices using pytest, contributing to a reliable codebase.  
- Utilized AWS for cloud hosting and S3 for scalable storage solutions, optimizing resource management.  
- Collaborated with data scientists to design and optimize data pipelines, streamlining workflows and improving data accessibility.  
- Engineered RESTful APIs for internal tools, facilitating seamless integration and communication between services.  

#### Few-Shot Rewrite
- Developed and maintained a high-performance web application using Django, enhancing user experience and scalability over a 3-year period.

- Deployed machine learning models to production environments, optimizing model serving infrastructure to handle ~10K requests per hour.

- Established robust unit testing framework with 85% coverage using pytest, seamlessly integrated into CI/CD pipeline to ensure code quality.

- Leveraged AWS for hosting and utilized S3 for efficient data storage, improving retrieval times by ~30% across multiple applications.

- Collaborated closely with the data team to design and implement data pipelines, enhancing data processing efficiency by ~25%.

- Built and optimized RESTful APIs for internal tools, supporting over 1,000 internal users and streamlining workflows across teams.


In [18]:
coverage = ask(
    f"""Analyse how well this resume matched the job description

Job Description:
{job_description}

Resume Bullets:
{zero_shot}

Return JSON with:
- "matched_keywords": list of JD keywords found in resume
- "missing_keywords": list of JD keywords NOT addressed
- "match_score": percentage (0-100)
- "suggestions": list of 2-3 improvements
""",
temperature = 0
)

display(Markdown(coverage))

```json
{
  "matched_keywords": [
    "5+ years of Python experience",
    "ML/AI deployment",
    "Docker",
    "Kubernetes",
    "strong testing practices",
    "pytest",
    "cloud platforms",
    "AWS"
  ],
  "missing_keywords": [
    "FastAPI",
    "Flask",
    "CI/CD"
  ],
  "match_score": 70,
  "suggestions": [
    "Include specific experience with FastAPI or Flask to align with the job requirements.",
    "Highlight any experience with CI/CD practices to demonstrate familiarity with continuous integration and deployment.",
    "Mention any experience with Docker and Kubernetes in more detail, especially in the context of ML pipelines."
  ]
}
```

---
## 5. Structured Output: Keyword Matching

Let's also check which job requirements are covered:

---
## Key Takeaways

| Technique | Effect |
|-----------|--------|
| **Zero-shot** | Quick but generic — good starting point |
| **Few-shot examples** | Controls output style and quality significantly |
| **Negative constraints** | "Do NOT fabricate" prevents hallucinated experience |
| **Keyword matching** | Structured JSON output to verify coverage |

> **Exercise:** Replace the sample job description with a real one you're interested in.
> Add your own work experience. Experiment with different few-shot examples to find
> the voice that works best for your industry.

---
**Next:** `module_03_ai_architecture` — Build production-ready AI applications